# Ex1 - GNSS-Denied Visual Navigation for Drones

**Pipeline:** DINOv2 (global retrieval) → XFeat (local matching) → RANSAC (homography) → GPS estimate

| Video | Altitude | Role |
|-------|----------|------|
| V1 (DJI Air 3) | 120 m @ 45° | Reference map (preprocessing) |
| V2 (DJI Air 3) | 50 m @ 45° | Query - navigation without GNSS |

> **Note:** V2 GPS (from SRT) is used **only** for error evaluation - never fed into the navigation pipeline.

## Step 0 - Setup

Mount Google Drive and define all file paths in one place.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import cv2
import numpy as np
import pandas as pd
import pickle
import re
import os

# --- All paths in one place ---
BASE = '/content/drive/MyDrive/ex1_new'

# Video 1 (120m) - preprocessing / map
V1_SRT    = f'{BASE}/DJI_20260427152226_0017_D.SRT'
V1_MP4    = f'{BASE}/DJI_20260427152226_0017_D.MP4'
V1_CSV    = f'{BASE}/srt_v1.csv'
V1_FRAMES = f'{BASE}/frames'

# Video 2 (50m) - navigation / test
V2_SRT    = f'{BASE}/DJI_20260427152735_0019_D.SRT'
V2_MP4    = f'{BASE}/DJI_20260427152735_0019_D.MP4'
V2_CSV    = f'{BASE}/srt_v2.csv'
V2_FRAMES = f'{BASE}/frames_v2'



## Reload (Optional)

If Steps 1–4 have already been run, reload saved files and skip ahead to Step 5b.

In [ ]:
# ── Reload (skip Steps 1-4 if already processed) ─────────────────────────────
import pickle, pandas as pd

df_v1  = pd.read_csv(V1_CSV)
df_v2  = pd.read_csv(V2_CSV)
df_map = pd.read_csv(f'{BASE}/map_frames.csv')

with open(f'{BASE}/map_db.pkl', 'rb') as f:
    map_db = pickle.load(f)

print(f'Loaded {len(map_db)} map frames from DB')

## Step 1 - Parse SRT Telemetry

Extract per-frame GPS, altitude, and camera metadata from DJI SRT files.

- **V1 SRT** → `srt_v1.csv` (reference map telemetry)
- **V2 SRT** → `srt_v2.csv` (query telemetry - used for evaluation only)

In [ ]:
def parse_srt(srt_path, output_csv):
    """Parse a DJI SRT file and save telemetry as CSV."""
    with open(srt_path, 'r', encoding='utf-8') as f:
        srt_content = f.read()

    # Split into subtitle blocks
    raw_entries = re.split(r'\n\n(?=\d+\n\d{2}:\d{2}:\d{2},\d{3})', srt_content.strip())
    subtitle_blocks = []
    for entry in raw_entries:
        lines = entry.split('\n', 2)
        if len(lines) >= 3:
            subtitle_blocks.append(lines[2])

    # Extract fields from each block
    patterns = {
        'FrameCnt':    r'FrameCnt:\s*(\d+)',
        'DiffTime_ms': r'DiffTime:\s*(\d+)ms',
        'Date':        r'(\d{4}-\d{2}-\d{2})',
        'Time':        r'\d{4}-\d{2}-\d{2}\s+(\d{2}:\d{2}:\d{2}\.\d+)',
        'rel_alt':     r'rel_alt:\s*([\d.]+)',
        'abs_alt':     r'abs_alt:\s*([\d.]+)',
        'iso':         r'iso:\s*(\d+)',
        'shutter':     r'shutter:\s*([\d/\.]+)',
        'fnum':        r'fnum:\s*([\d.]+)',
        'ev':          r'ev:\s*([-\d.]+)',
        'color_md':    r'color_md:\s*(\w+)',
        'focal_len':   r'focal_len:\s*([\d.]+)',
        'latitude':    r'latitude:\s*([-\d.]+)',
        'longitude':   r'longitude:\s*([-\d.]+)',
        'ct':          r'\[ct:\s*(\d+)\]',
    }

    def parse_block(text):
        row = {}
        for key, pattern in patterns.items():
            match = re.search(pattern, text)
            row[key] = match.group(1) if match else None
        return row

    df = pd.DataFrame([parse_block(b) for b in subtitle_blocks])

    # Convert types
    for col in ['FrameCnt', 'DiffTime_ms', 'iso', 'ct']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    for col in ['rel_alt', 'abs_alt', 'fnum', 'ev', 'focal_len', 'latitude', 'longitude']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df.to_csv(output_csv, index=False)
    print(f'Parsed {len(df)} frames → saved to {output_csv}')
    print(df[['FrameCnt', 'rel_alt', 'latitude', 'longitude']].head(3))
    return df

# Parse both videos
print('=== Video 1 (120m) ===')
df_v1 = parse_srt(V1_SRT, V1_CSV)
print()
print('=== Video 2 (50m) ===')
df_v2 = parse_srt(V2_SRT, V2_CSV)

## Step 2 - Compute Center GPS & Filter Takeoff

For each frame, compute the GPS coordinate of the camera center pixel on the ground.

**Geometry:** With fixed 45° gimbal angle from nadir:
$$d = h \times \tan(45°) = h$$
The center point is exactly `altitude` metres ahead of the drone.

**Takeoff filter:** Derivative-based - discard frames where altitude is still rising (derivative > 0.5 m/frame for a window of 15 frames, and altitude < 95% of cruise altitude).

**Heading:** Derived from consecutive GPS positions, smoothed in sin/cos space (median filter, window=61) to handle 0°/360° wraparound at turns.

In [ ]:
# ── Step 2 — Compute center GPS + filter takeoff ─────────────────────────────

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EARTH_RADIUS_M      = 6_378_137.0
TAN_45              = math.tan(math.radians(45.0))
ALT_DERIV_THRESHOLD = 0.5
STABLE_WINDOW       = 15


def find_stable_start(alt_series):
    alts        = alt_series.values
    deriv       = np.abs(np.gradient(alts))
    cruise_alt  = np.percentile(alts, 95)
    near_cruise = alts >= cruise_alt * 0.95

    consecutive = 0
    for i, (d, close) in enumerate(zip(deriv, near_cruise)):
        if d < ALT_DERIV_THRESHOLD and close:
            consecutive += 1
            if consecutive >= STABLE_WINDOW:
                return i - STABLE_WINDOW + 1
        else:
            consecutive = 0

    print('  Warning: no stable cruise found — using full video.')
    return 0


def compute_heading_strictly_moving(lats, lons, min_displacement_m = 0.2):

    headings = np.zeros(len(lats))

    for i in range(len(lats) - 1):
        dlat_m = (lats[i+1] - lats[i]) * (EARTH_RADIUS_M * math.pi / 180)
        dlon_m = (lons[i+1] - lons[i]) * (EARTH_RADIUS_M * math.pi / 180) * math.cos(math.radians(lats[i]))
        displacement = math.sqrt(dlat_m**2 + dlon_m**2)

        if displacement > min_displacement_m:
            headings[i] = math.degrees(math.atan2(dlon_m, dlat_m)) % 360
        else:
            headings[i] = np.nan

    headings[-1] = np.nan

    headings_series = pd.Series(headings).ffill().bfill()

    rads = np.radians(headings_series.values)
    sin_h = np.sin(rads)
    cos_h = np.cos(rads)

    sin_smooth = pd.Series(sin_h).rolling(window=91, center=True, min_periods=1).mean().values
    cos_smooth = pd.Series(cos_h).rolling(window=91, center=True, min_periods=1).mean().values

    headings = np.degrees(np.arctan2(sin_smooth, cos_smooth)) % 360

    return headings


def compute_center_gps(lat, lon, alt_m, heading_deg):
    offset_m    = alt_m * TAN_45
    heading_rad = math.radians(heading_deg)
    delta_lat   = (offset_m * math.cos(heading_rad) / EARTH_RADIUS_M) * (180 / math.pi)
    delta_lon   = (offset_m * math.sin(heading_rad) /
                  (EARTH_RADIUS_M * math.cos(math.radians(lat)))) * (180 / math.pi)
    return lat + delta_lat, lon + delta_lon, offset_m


def process_video(df, label):
    print(f'=== {label} ===')
    print(f'  Frames before filter : {len(df)}')

    stable_idx = find_stable_start(df['rel_alt'])
    df = df.iloc[stable_idx:].copy().reset_index(drop=True)

    print(f'  Takeoff cut at index : {stable_idx}')
    print(f'  Frames after filter  : {len(df)}')
    print(f'  Alt mean / std       : {df["rel_alt"].mean():.1f} / {df["rel_alt"].std():.2f} m')

    # תיקון שם הפונקציה: קריאה ישירה לפונקציה הנכונה compute_heading_strictly_moving
    df['heading_deg'] = compute_heading_strictly_moving(
        df['latitude'].values, df['longitude'].values
    )

    results = [
        compute_center_gps(r.latitude, r.longitude, r.rel_alt, r.heading_deg)
        for r in df.itertuples()
    ]
    df['center_lat'], df['center_lon'], df['ground_offset_m'] = zip(*results)

    print(df[['FrameCnt', 'rel_alt', 'heading_deg',
              'center_lat', 'center_lon']].head(3).to_string(index=False))
    print()
    return df


df_v1 = process_video(df_v1, 'Video 1 — 100 m')
df_v2 = process_video(df_v2, 'Video 2 — 30 m')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, df, label in [(axes[0], df_v1, 'V1 100m'), (axes[1], df_v2, 'V2 30m')]:
    ax.plot(df['longitude'],  df['latitude'],   'b.-', markersize=2, alpha=0.5, label='drone GPS')
    ax.plot(df['center_lon'], df['center_lat'], 'r.-', markersize=2, alpha=0.5, label='center point')
    ax.set_title(label)
    ax.legend()
    ax.set_xlabel('longitude')
    ax.set_ylabel('latitude')
plt.tight_layout()
plt.show()

## Step 3 - Index Existing Map Frames

V1 frames were pre-extracted (every 10th frame → ~688 frames). This cell indexes them against the SRT telemetry to build `map_frames.csv` with center GPS per frame.

In [ ]:
# ── Step 3 - Index existing frames ───────────────────────────────────────────

df_v1['frame_num'] = df_v1['FrameCnt'].astype(int) - 1
frame_lookup = df_v1.set_index('frame_num')

existing = sorted([f for f in os.listdir(V1_FRAMES) if f.endswith('.jpg')])

extracted = []
for fname in existing:
    # filename is e.g. 'frame_02740.jpg' → frame_num = 2740
    frame_num = int(fname.replace('frame_', '').replace('.jpg', ''))

    if frame_num not in frame_lookup.index:
        continue

    row = frame_lookup.loc[frame_num]
    extracted.append({
        'frame_num'  : frame_num,
        'FrameCnt'   : row['FrameCnt'],
        'rel_alt'    : row['rel_alt'],
        'latitude'   : row['latitude'],
        'longitude'  : row['longitude'],
        'heading_deg': row['heading_deg'],
        'center_lat' : row['center_lat'],
        'center_lon' : row['center_lon'],
        'filename'   : fname,
    })

df_map = pd.DataFrame(extracted).reset_index(drop=True)
df_map.to_csv(f'{BASE}/map_frames.csv', index=False)

print(f'Indexed : {len(df_map)} frames')
print(df_map[['frame_num', 'rel_alt', 'center_lat', 'center_lon']].head(5).to_string(index=False))

## Step 4a - Install XFeat

Clone the official XFeat repository (CVPR 2024) and install FAISS.

- **XFeat paper:** https://arxiv.org/abs/2404.19174
- **XFeat code:** https://github.com/verlab/accelerated_features

In [ ]:
    !git clone https://github.com/verlab/accelerated_features.git
    !pip install faiss-cpu

In [ ]:
import sys
sys.path.append('/content/accelerated_features')
from modules.xfeat import XFeat
print('XFeat imported OK')

## Step 4b - XFeat Feature Extraction (Map Database)

Run XFeat `detectAndCompute` on all 688 map frames.

- `top_k=2048` keypoints per frame
- 64-dim descriptors stored per keypoint
- Saved to `map_db.pkl`

In [ ]:
# ── Step 4 - XFeat feature extraction on map frames ──────────────────────────

import sys
sys.path.append('/content/accelerated_features')
from modules.xfeat import XFeat

import torch
import cv2
import numpy as np
import pickle
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

xfeat = XFeat()
xfeat.eval()

MAP_DB_PATH = f'{BASE}/map_db.pkl'

map_db = []
failed = 0

for i, row in df_map.iterrows():
    fpath = os.path.join(V1_FRAMES, row['filename'])
    img   = cv2.imread(fpath)
    if img is None:
        failed += 1
        continue

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    with torch.no_grad():
        out = xfeat.detectAndCompute(img_rgb, top_k=2048)

    map_db.append({
        'frame_num'  : row['frame_num'],
        'center_lat' : row['center_lat'],
        'center_lon' : row['center_lon'],
        'rel_alt'    : row['rel_alt'],
        'heading_deg': row['heading_deg'],
        'keypoints'  : out[0]['keypoints'].cpu().numpy(),
        'descriptors': out[0]['descriptors'].cpu().numpy(),
        'filename'   : row['filename'],
    })

    if (i + 1) % 50 == 0:
        print(f'  {i + 1}/{len(df_map)} frames processed...')

with open(MAP_DB_PATH, 'wb') as f:
    pickle.dump(map_db, f)

print(f'\nDone: {len(map_db)} frames in DB ({failed} failed)')
print(f'Saved to: {MAP_DB_PATH}')
print(f'Sample — frame {map_db[0]["frame_num"]}: '
      f'{map_db[0]["keypoints"].shape[0]} keypoints')

## Save Intermediate Results

Save all dataframes and the map database so the notebook can be restarted without re-running Steps 1–4.

In [ ]:
# Save df_map and df_v1, df_v2 as well so you can reload without rerunning
df_map.to_csv(f'{BASE}/map_frames.csv', index=False)
df_v1.to_csv(V1_CSV, index=False)
df_v2.to_csv(V2_CSV, index=False)

print('Saved:')
print(f'  {BASE}/map_db.pkl      ← XFeat features')
print(f'  {BASE}/map_frames.csv  ← frame metadata')
print(f'  {V1_CSV}               ← V1 telemetry')
print(f'  {V2_CSV}               ← V2 telemetry')

## Step 5a - Build XFeat Mean-Pool FAISS Index (Baseline)

Aggregate XFeat keypoint descriptors per frame using mean pooling, then build a FAISS inner-product index for fast retrieval.

> This baseline is superseded by the DINOv2 index in Step 5b.

In [ ]:
# ── Step 5 - Build FAISS index ───────────────────────────────────────────────

import faiss
import numpy as np

FAISS_PATH = f'{BASE}/faiss_index.bin'
META_PATH  = f'{BASE}/faiss_meta.pkl'

# Build per-frame descriptor by mean-pooling keypoint descriptors
descriptors_matrix = []
meta = []

for entry in map_db:
    desc = entry['descriptors']           # (N, 64)
    mean_desc = desc.mean(axis=0)         # (64,)
    # L2-normalize for cosine similarity
    mean_desc /= (np.linalg.norm(mean_desc) + 1e-8)
    descriptors_matrix.append(mean_desc)

    meta.append({
        'frame_num'  : entry['frame_num'],
        'center_lat' : entry['center_lat'],
        'center_lon' : entry['center_lon'],
        'rel_alt'    : entry['rel_alt'],
        'heading_deg': entry['heading_deg'],
        'filename'   : entry['filename'],
    })

descriptors_matrix = np.array(descriptors_matrix, dtype=np.float32)  # (688, 64)
print(f'Descriptor matrix: {descriptors_matrix.shape}')

# Build flat L2 index (exact search — fast enough for 688 frames)
index = faiss.IndexFlatIP(64)   # inner product = cosine sim after L2 norm
index.add(descriptors_matrix)

faiss.write_index(index, FAISS_PATH)

with open(META_PATH, 'wb') as f:
    pickle.dump(meta, f)

print(f'FAISS index: {index.ntotal} vectors')
print(f'Saved to: {FAISS_PATH}')

## Step 5b - DINOv2 Global Retrieval Index

Replace mean-pooled XFeat with DINOv2 ViT-S/14 embeddings for global retrieval. DINOv2 features generalize robustly across altitude, season, and viewpoint changes without any fine-tuning.

- Each map frame → 384-dim L2-normalized embedding
- FAISS `IndexFlatIP` (exact cosine similarity search)
- Saved to `dino_faiss.bin` + `dino_meta.pkl`

**DINOv2 paper:** https://arxiv.org/abs/2304.07193  
**DINOv2 code:** https://github.com/facebookresearch/dinov2

In [ ]:
# ── Step 5b - Replace FAISS with DINOv2 retrieval ────────────────────────────
!pip install timm -q

import torch
import torchvision.transforms as T

# Load DINOv2 (small variant — fast on Colab CPU/GPU)
dinov2 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
dinov2.eval()

transform = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]),
])

def get_dino_embedding(img_rgb):
    tensor = transform(img_rgb).unsqueeze(0)
    with torch.no_grad():
        emb = dinov2(tensor)
    emb = emb / (emb.norm(dim=1, keepdim=True) + 1e-8)
    return emb.squeeze().numpy().astype(np.float32)

# Build DINOv2 embeddings for all map frames
print('Building DINOv2 map embeddings...')
dino_embeddings = []
dino_meta       = []

for i, entry in enumerate(map_db):
    img     = cv2.imread(os.path.join(V1_FRAMES, entry['filename']))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    emb     = get_dino_embedding(img_rgb)
    dino_embeddings.append(emb)
    dino_meta.append({
        'frame_num'  : entry['frame_num'],
        'center_lat' : entry['center_lat'],
        'center_lon' : entry['center_lon'],
        'rel_alt'    : entry['rel_alt'],
        'heading_deg': entry['heading_deg'],
        'filename'   : entry['filename'],
    })
    if (i+1) % 50 == 0:
        print(f'  {i+1}/{len(map_db)} done...')

dino_matrix = np.array(dino_embeddings, dtype=np.float32)

# Build FAISS index on DINOv2 embeddings
dino_index = faiss.IndexFlatIP(dino_matrix.shape[1])
dino_index.add(dino_matrix)

faiss.write_index(dino_index, f'{BASE}/dino_faiss.bin')
with open(f'{BASE}/dino_meta.pkl', 'wb') as f:
    pickle.dump(dino_meta, f)

print(f'DINOv2 FAISS index: {dino_index.ntotal} vectors of dim {dino_matrix.shape[1]}')

In [ ]:
faiss.write_index(dino_index, f'{BASE}/dino_faiss.bin')
with open(f'{BASE}/dino_meta.pkl', 'wb') as f:
    pickle.dump(dino_meta, f)

## Step 6 - Navigation: V2 Frames → GPS Estimate

For each V2 query frame (no GNSS):

1. **Scale normalization** - resize by 50/120 ≈ 0.417 to match V1's ground scale
2. **DINOv2 retrieval** - FAISS cosine search → top-K=5 candidate map frames
3. **XFeat matching** - mutual nearest-neighbor matching between query and each candidate
4. **RANSAC homography** - `cv2.findHomography` with 5px threshold → count inliers
5. **GPS prediction** - best candidate (most inliers) center GPS, if inliers ≥ 10

Results saved to `navigation_results.csv`.

In [ ]:
# ── Step 6 - Navigation with DINOv2 retrieval + XFeat matching ───────────────

TOP_K_RETRIEVAL = 5
MIN_INLIERS     = 10
SCALE_FACTOR    = 50 / 120

dino_index = faiss.read_index(f'{BASE}/dino_faiss.bin')
with open(f'{BASE}/dino_meta.pkl', 'rb') as f:
    dino_meta = pickle.load(f)

map_db_lookup = {e['frame_num']: e for e in map_db}
dino_meta_lookup = {e['frame_num']: e for e in dino_meta}

df_v2['frame_num'] = df_v2['FrameCnt'].astype(int) - 1
v2_lookup = df_v2.set_index('frame_num')

v2_files = sorted([f for f in os.listdir(V2_FRAMES) if f.endswith('.jpg')])
print(f'V2 frames to process: {len(v2_files)}')


def rescale_frame(img, scale):
    h, w    = img.shape[:2]
    new_w   = int(w * scale)
    new_h   = int(h * scale)
    resized = cv2.resize(img, (new_w, new_h))
    top     = (h - new_h) // 2
    left    = (w - new_w) // 2
    padded  = np.zeros_like(img)
    padded[top:top+new_h, left:left+new_w] = resized
    return padded


results = []

for i, fname in enumerate(v2_files):
    frame_num = int(fname.replace('frame_', '').replace('.jpg', ''))

    if frame_num not in v2_lookup.index:
        continue

    row     = v2_lookup.loc[frame_num]
    img     = cv2.imread(os.path.join(V2_FRAMES, fname))
    if img is None:
        continue

    img_rgb        = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_rgb_scaled = rescale_frame(img_rgb, SCALE_FACTOR)

    # 1. DINOv2 retrieval on scaled frame
    dino_emb = get_dino_embedding(img_rgb_scaled)
    _, indices = dino_index.search(dino_emb.reshape(1, -1), TOP_K_RETRIEVAL)

    # 2. XFeat on scaled frame
    with torch.no_grad():
        query_out = xfeat.detectAndCompute(img_rgb_scaled, top_k=2048)

    query_desc = query_out[0]['descriptors'].cpu().numpy()
    query_kpts = query_out[0]['keypoints'].cpu().numpy()

    # 3. Match + RANSAC against top-K candidates
    best_inliers = 0
    best_meta    = None

    for idx in indices[0]:
        if idx < 0 or idx >= len(dino_meta):
            continue

        cand_frame = dino_meta[idx]['frame_num']
        cand_entry = map_db_lookup.get(cand_frame)
        if cand_entry is None:
            continue

        q_t    = torch.tensor(query_desc)
        c_t    = torch.tensor(cand_entry['descriptors'])
        q_norm = q_t / (q_t.norm(dim=1, keepdim=True) + 1e-8)
        c_norm = c_t / (c_t.norm(dim=1, keepdim=True) + 1e-8)
        sim    = q_norm @ c_norm.T

        best_q = sim.argmax(dim=1)
        best_c = sim.argmax(dim=0)
        mutual = (best_c[best_q] == torch.arange(len(query_desc)))

        q_idx = torch.where(mutual)[0].numpy()
        c_idx = best_q[q_idx].numpy()

        if len(q_idx) < 4:
            continue

        pts_q = query_kpts[q_idx].astype(np.float32)
        pts_c = cand_entry['keypoints'][c_idx].astype(np.float32)

        H, mask = cv2.findHomography(pts_q, pts_c, cv2.RANSAC, 5.0)
        if mask is None:
            continue

        n_inliers = int(mask.sum())
        if n_inliers > best_inliers:
            best_inliers = n_inliers
            best_meta    = dino_meta[idx]

    results.append({
        'frame_num'    : frame_num,
        'gt_lat'       : row['center_lat'],
        'gt_lon'       : row['center_lon'],
        'n_inliers'    : best_inliers,
        'matched_frame': best_meta['frame_num'] if best_meta else None,
        'pred_lat'     : best_meta['center_lat'] if best_meta and best_inliers >= MIN_INLIERS else None,
        'pred_lon'     : best_meta['center_lon'] if best_meta and best_inliers >= MIN_INLIERS else None,
    })

    if (i + 1) % 20 == 0:
        print(f'  {i+1}/{len(v2_files)} done...')

df_results = pd.DataFrame(results)
df_results.to_csv(f'{BASE}/navigation_results.csv', index=False)

matched = df_results['pred_lat'].notna().sum()
print(f'\nMatched : {matched}/{len(df_results)} frames')
print(df_results[['frame_num', 'gt_lat', 'pred_lat', 'n_inliers']].head(10).to_string(index=False))

## Step 7 - Error Analysis

Evaluate predicted positions against V2 SRT ground truth using Haversine distance.


In [ ]:
# ── Step 7 - Error analysis ───────────────────────────────────────────────────
import matplotlib.pyplot as plt

def haversine_m(lat1, lon1, lat2, lon2):
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a    = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
    return 6_378_137.0 * 2 * np.arcsin(np.sqrt(a))

df_eval = df_results.dropna(subset=['pred_lat', 'pred_lon']).copy()
df_eval['error_m'] = df_eval.apply(
    lambda r: haversine_m(r.gt_lat, r.gt_lon, r.pred_lat, r.pred_lon), axis=1
)

print(f'Matched frames : {len(df_eval)}/{len(df_results)}')
print(f'Error mean     : {df_eval["error_m"].mean():.1f} m')
print(f'Error median   : {df_eval["error_m"].median():.1f} m')
print(f'Error std      : {df_eval["error_m"].std():.1f} m')
print(f'Error < 50m    : {(df_eval["error_m"] < 50).sum()} frames')
print(f'Error < 100m   : {(df_eval["error_m"] < 100).sum()} frames')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_eval['error_m'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_xlabel('error (m)')
axes[0].set_ylabel('frames')
axes[0].set_title('Position error distribution')

axes[1].plot(df_eval['gt_lon'],   df_eval['gt_lat'],   'b.-', markersize=3, label='ground truth')
axes[1].plot(df_eval['pred_lon'], df_eval['pred_lat'], 'r.-', markersize=3, label='predicted')
axes[1].set_xlabel('longitude')
axes[1].set_ylabel('latitude')
axes[1].set_title('GT vs predicted path')
axes[1].legend()

plt.tight_layout()
plt.show()

## Step 8 - Interactive Map (Folium)

Visualize ground truth path (green) vs predicted positions (red) on a satellite basemap.

In [ ]:
# ── Step 8 - Interactive map ──────────────────────────────────────────────────
!pip install folium -q
import folium

# Reload from CSV to match Step 7
df_results = pd.read_csv(f'{BASE}/navigation_results.csv')
df_results['error_m'] = df_results.apply(
    lambda r: haversine_m(r.gt_lat, r.gt_lon, r.pred_lat, r.pred_lon)
    if pd.notna(r.pred_lat) else None, axis=1
)

center_lat = df_results['gt_lat'].mean()
center_lon = df_results['gt_lon'].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=17)

# Ground truth path (green)
actual_coords = list(zip(df_results['gt_lat'], df_results['gt_lon']))
folium.PolyLine(actual_coords, color='green', weight=3,
                tooltip='Actual GPS path (V2 SRT)').add_to(m)

# Predicted path (red) — only matched frames
pred_df = df_results.dropna(subset=['pred_lat', 'pred_lon'])
pred_coords = list(zip(pred_df['pred_lat'], pred_df['pred_lon']))
folium.PolyLine(pred_coords, color='red', weight=3,
                tooltip='Estimated path (DINOv2 + XFeat)').add_to(m)

# Markers on ground truth points
for _, row in df_results.iterrows():
    folium.CircleMarker(
        location=[row['gt_lat'], row['gt_lon']],
        radius=3, color='green', fill=True
    ).add_to(m)

# Legend
legend = '''
<div style="position: fixed; bottom: 50px; left: 50px; z-index: 1000;
            background-color: white; padding: 10px; border-radius: 5px;">
    <b>Legend</b><br>
    <span style="color: green;">●</span> Actual GPS path (from SRT)<br>
    <span style="color: red;">●</span> Estimated path (DINOv2 + XFeat + RANSAC)
</div>
'''
m.get_root().html.add_child(folium.Element(legend))

map_path = f'{BASE}/navigation_map.html'
m.save(map_path)
print(f'Map saved to {map_path}')
print(f'Matched frames : {len(pred_df)}/354')
print(f'Average error  : {pred_df["error_m"].mean():.1f} m')
m